# Time-resolved scattered-light spectrum (tSPM)

A 351 nm laser heats a plasma. Some of the light is scattered back by **laser-plasma instabilities**,
and each instability leaves a fingerprint at its own wavelength:

| Feature | Wavelength | Origin |
|---|---|---|
| SRS | ~ 600–660 nm | stimulated Raman scattering from plasma below quarter-critical density |
| ω/2 | ~ 702–712 nm (2 × 351 nm) | two-plasmon decay / SRS near **quarter-critical density** $n_c/4$ |

A spectrometer disperses the light by wavelength, and a **streak camera** sweeps it in time.
One shot gives one 2-D image: wavelength against time.
In this session you take that image from the raw file to a publication-style figure, and compare the choices you make on the way.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
import tspm   # the helper functions for this session live in tspm.py

DATA = 'data/time-resolved spec/'

## Step 1: look at the raw data

The `.dac` file is a text table exported by the streak-camera software.
Left: the image as the camera saw it, in pixels. Right: the histogram of all pixel values.

**Think about it.**
1. What is the large circle? (Hint: the streak tube's photocathode is round.)
2. The histogram has a sharp peak near 110 counts. Is that light, or the camera?

In [ ]:
raw = tspm.load_dac(DATA + '20210814110.dac')

## Step 2: the time and wavelength axes

The camera software has already calibrated the axes and stored them in the file:
the first row is the wavelength of every pixel row, and the first column is the time of every pixel column.
We just read them and use them.

Left plot: how many ps one pixel column represents. It is **not constant**: the camera's sweep slows down by about 10 % across the screen.
A straight line through the first and last time would be wrong by up to 0.12 ns in the middle, so we keep the time of every pixel as stored.

**Think about it.** The time axis starts at a negative value. Does $t = 0$ mean anything physical yet?

In [ ]:
tspm.show_axes(raw)

## Step 3: preprocessing

A 3 × 3 median filter removes single hot pixels; a Gaussian filter then smooths the photon noise. The plots zoom in on the ω/2 feature.

**Exercise.**
- Change `sigma` to 0 and to 5. What do you gain and what do you lose?
- The two thin arcs in the top-left of the map are the edge of the photocathode, not plasma light.
  The original analysis blanked a box (692–765 nm, file time −1.45 to 0.4 ns) to hide them.
  Set `I[...] = 0` in that box yourself and redo step 8. What happens to the onset of ω/2? Is the box safe?

In [ ]:
clean = tspm.preprocess(raw, t_dark=-1.5, median=3, sigma=2,
                        vmin=10, xlim=(-0.5, 3.5), ylim=(700, 715))

## Step 4a: compute the group-velocity dispersion (GVD)

The light travels to the spectrometer through **33 m of fused-silica fiber**.
Glass slows short wavelengths more than long ones: the group index is
$N_g(\lambda) = n - \lambda\,\mathrm{d}n/\mathrm{d}\lambda$, with $n(\lambda)$ from the Sellmeier equation.
So different wavelengths that left the plasma at the same moment arrive at different times:

$$\Delta t(\lambda) = \frac{L}{c}\,\bigl[N_g(\lambda) - N_g(\lambda_\mathrm{ref})\bigr]$$

We take the shortest wavelength in the window as the reference, so every other wavelength has $\Delta t < 0$: it arrived early.

**Think about it.** The left plot shows $N_g$ falling with wavelength. Which colour reaches the camera first?

In [ ]:
delay = tspm.gvd_delay(clean.lam, fiber_m=33)

## Step 4b: correct the data

Each wavelength row is shifted later by its own $|\Delta t|$: corrected$(t)$ = measured$(t + \Delta t)$.
The shift is not a whole number of pixels, so we interpolate between pixels.
Times that move in from outside the recorded window have no data and are left blank.

**Think about it.** The delay across the window is about 1 ns. The whole laser pulse is about 4.7 ns.
Which conclusions about the *timing* of SRS versus ω/2 would be wrong without this correction?

In [ ]:
gvd = tspm.gvd_correct(clean, delay, xlim=(-1, 5), ylim=(580, 764))

## Step 5: linear or logarithmic colour scale?

The signal spans more than two orders of magnitude.

In [ ]:
tspm.plot_maps(gvd, [dict(title='linear', log=False),
                     dict(title='logarithmic', log=True)],
               vmax=3e4, xlim=(-1, 5), ylim=(600, 740))

**Think about it.** On the linear map, which feature almost disappears? On the log map, where does the noise start to show?
Which scale would you choose to show that *all three* features exist, and which to compare their *peak brightness*?

## Step 6: which colour map?

The same data, the same log scale, four colour maps. `jet` was used in the original paper figure;
`WHITE_JET` is a custom white-to-red map from the same analysis.

In [ ]:
# Other colour maps to try (any name from matplotlib works; full list: plt.colormaps()):
#   perceptually uniform, sequential: 'viridis', 'plasma', 'inferno', 'magma', 'cividis'
#   single hue, light to dark:        'Greys', 'Purples', 'Blues', 'Greens', 'Oranges', 'Reds'
#   multi-hue, light to dark:         'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
#                                     'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn'
#   grey and "heat" maps:             'gray', 'binary', 'bone', 'pink', 'hot', 'afmhot', 'gist_heat', 'copper'
#   other sequential:                 'cubehelix', 'CMRmap', 'gnuplot', 'gnuplot2', 'cool', 'Wistia'
#   rainbow-like (not uniform):       'jet', 'turbo', 'rainbow', 'nipy_spectral', 'gist_rainbow', 'gist_ncar'
#   diverging (for signed data, centred on a reference; not for intensity):
#                                     'coolwarm', 'bwr', 'seismic', 'RdBu', 'RdYlBu', 'Spectral', 'PiYG', 'BrBG'
#   cyclic (for angles or phase; not for intensity): 'twilight', 'twilight_shifted', 'hsv'
# Add '_r' to any name to reverse it, e.g. 'magma_r'.
tspm.plot_maps(gvd, [dict(title='jet', cmap='jet'),
                     dict(title='WHITE_JET (custom)', cmap=tspm.WHITE_JET),
                     dict(title='viridis', cmap='viridis'),
                     dict(title='gray_r', cmap='gray_r')],
               xlim=(-1, 5), ylim=(600, 740))

**Think about it.**
1. In `jet`, the yellow-cyan band looks like a sharp edge. Is there a sharp edge in the data?
2. Which map would still work printed in black and white, or for a colour-blind reader?
3. `viridis` gets brighter monotonically with the value. Why does that matter for reading intensity?

## Step 7a: load and look at the laser pulse

Eight beams hit the target. Their power was recorded by oscilloscopes.
We put $t = 0$ at the start of the 4.67 ns pulse (4.67 ns before its falling edge),
normalise each beam to its own peak, and average them.

In [ ]:
t_laser, P = tspm.load_laser(DATA + '20210814110_2X_*.csv', pulse_ns=4.67)
laser = P.mean(axis=0)

## Step 7b: put the laser on the spectrum

The streak camera's time zero is arbitrary, so we shift the spectrum by `T0` to put it on the laser's time axis.
The original analysis used `T0 = 1.05` ns. The white curve is the laser pulse, scaled to fit in the 605–660 nm range of the plot.

In [ ]:
T0 = 1.05                           # ns, spectrum time -> laser time
spec = replace(gvd, t=gvd.t + T0)
CMAP = 'viridis'                    # try any colour map from step 6
LINE = 'w'                          # laser line colour; use a dark one (e.g. 'k') on maps that start white
tspm.overlay_laser(spec, t_laser, laser, band=(605, 660), xlim=(0, 6), ylim=(605, 740), cmap=CMAP, color=LINE)

**Think about it.** Scattered light needs the laser. When the laser switches off, the SRS signal should stop within the plasma response time.
Does it, with `T0 = 1.05`? Try other values of `T0`. What would you need to *measure* `T0` instead of choosing it?

## Step 8: lineouts of different wavelength bands

Average the map over a wavelength band to get one time history per instability, then compare their timing with the laser.
Left: the map with a box around each band. Right: the lineouts, in the same colours.

In [ ]:
BANDS = {'SRS': (610, 650), 'broad, 680 nm': (670, 695), 'ω/2': (705, 712)}
tspm.lineouts(spec, BANDS, t_laser, laser, xlim=(0, 6), ylim=(600, 740))

**Think about it.**
1. Which instability starts first? At what laser power (as a fraction of the peak)?
2. Make the ω/2 band narrower, e.g. (707, 710). Does its onset time change? Why?
3. Repeat this step with the map from step 3 (`replace(clean, t=clean.t + T0)`, before the GVD correction). How much do the onset times move?